# Run the VDocRAG demo

Launches the actual Gradio app (`app.py`) with the real, GPU-loaded retriever/
generator. Requires Steps 1–3 (`00_smoke_test.ipynb`, `01_wrapper_test.ipynb`)
to have already confirmed this configuration works on your hardware — this
notebook doesn't re-diagnose problems, it just runs the app.

If this notebook fails somewhere Steps 1–3 didn't, that's a real gap between
the tested path and `app.py`'s actual usage — worth reporting, not chasing
cell-by-cell fixes here first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/vdocrag-project/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
!apt-get install -y poppler-utils -q

!pip install -q --upgrade pip
# pinned to the last 4.x release before transformers' 5.0 major version --
# see docs/implementation_plan.md Section 4.6b-4.6f for why this exact version.
# torch pinned explicitly to whatever Colab's runtime already ships
# (confirmed via `import torch; torch.__version__` on a fresh runtime --
# check this yourself if Colab's default version has since changed).
# Without this pin, one of accelerate/bitsandbytes/peft/gradio's own
# torch version requirements can cause pip to silently swap Colab's
# working CUDA-enabled torch for a CPU-only build from plain PyPI --
# confirmed happening here: nvidia-smi showed a T4 present and a fresh
# runtime's torch.cuda.is_available() was True, but after installing our
# other deps it became False (`Torch not compiled with CUDA enabled`).
!pip install -q torch==2.11.0 transformers==4.57.3 accelerate bitsandbytes peft pillow
!pip install -q pdf2image faiss-cpu gradio pandas

# NTT's package installed by URL every session, never vendored -- see docs/licenses.md
!pip install -q git+https://github.com/nttmdlab-nlp/VDocRAG.git

# huggingface-hub had a major version bump to 1.0 recently; transformers==4.57.3
# hard-requires <1.0 and crashes at import time otherwise (confirmed: this is a
# real, open issue -- huggingface/transformers#42670). gradio's dependency chain
# pulls in the newest huggingface-hub by default, which silently breaks
# transformers after gradio installs. gradio itself tolerates the older version
# fine (no hard requirement, just a resolver warning) -- so force it back down
# as the LAST install step, after everything else.
!pip install -q "huggingface-hub==0.35.3"

!rm -rf /content/repo
!git clone https://github.com/thejainamjain/vdocrag-project.git /content/repo
import sys
sys.path.insert(0, '/content/repo')

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "CUDA not available -- torch was likely silently swapped for a CPU-only build "
    "during the installs above (a known risk here: accelerate/bitsandbytes/peft/gradio "
    "can each pull a torch version that doesn't match the pin). Check `!nvidia-smi` "
    "shows a GPU, check `torch.__version__` for a '+cpu' suffix, and if so, re-run "
    "with an updated torch==<version> pin matching what a completely fresh runtime "
    "reports before any installs."
)
print(f"CUDA OK: {torch.cuda.get_device_name(0)}, torch {torch.__version__}")

## Load the app

This one cell does everything Steps 1–3 verified piece by piece: loads the
quantized base model with the confirmed `eager` + `num_crops=4` config, attaches
both LoRA adapters with shared-base hot-swap, wraps them in the retriever/
generator classes, and builds the Gradio Blocks UI around them. Expect a few
minutes on first run (model download); fast on later runs (Drive-cached weights).

In [ ]:
from app import main
demo = main()

## Launch

Kept as its own cell (separate from the load step above) so the notebook can
be re-launched — e.g. after a Gradio crash or to pick up a UI tweak in `app.py`
— without reloading the multi-GB model each time. `share=True` gives a public
URL usable from your Mac's browser; this cell blocks until you interrupt it.

In [ ]:
demo.launch(share=True, debug=True)